In [16]:
import torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

In [17]:
from datasets import Dataset

In [18]:
train_data = Dataset.from_dict({
    "text": ["这部电影太棒了", "非常无聊，浪费时间", "演技在线，值得推荐", "剧情老套，没意思"],
    "label": [1, 0, 1, 0],  # 1=正面, 0=负面
})

In [19]:
model_name = "bert-base-chinese"

In [20]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [21]:
def tokenize_fn(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=64)

In [22]:
train_dataset = train_data.map(tokenize_fn, batched=True)

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [23]:
training_args = TrainingArguments(
    output_dir="./output",
    num_train_epochs=30,
    per_device_train_batch_size=8,
    learning_rate=2e-5,
    logging_steps=10,
    save_strategy="epoch"
)

In [24]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

In [25]:
trainer.train()

Step,Training Loss
10,0.317800
20,0.078300
30,0.029600


TrainOutput(global_step=30, training_loss=0.14191023111343384, metrics={'train_runtime': 30.9283, 'train_samples_per_second': 3.88, 'train_steps_per_second': 0.97, 'total_flos': 3946665830400.0, 'train_loss': 0.14191023111343384, 'epoch': 30.0})

In [26]:
trainer.save_model("./my-finetuned-bert")
tokenizer.save_pretrained("./my-finetuned-bert")

('./my-finetuned-bert\\tokenizer_config.json',
 './my-finetuned-bert\\special_tokens_map.json',
 './my-finetuned-bert\\vocab.txt',
 './my-finetuned-bert\\added_tokens.json',
 './my-finetuned-bert\\tokenizer.json')

In [27]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis", model="./my-finetuned-bert")

Device set to use cuda:0


In [30]:
print(classifier("这部电影真是太棒了"))

[{'label': 'LABEL_1', 'score': 0.9674996733665466}]


In [29]:
print(classifier("你真丑"))

[{'label': 'LABEL_1', 'score': 0.8456034660339355}]


In [31]:
import evaluate

accuracy_metric = evaluate.load("accuracy")

In [32]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits), dim=-1)

    return accuracy_metric.compute(predictions=predictions, references=labels)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    compute_metrics=compute_metrics
)

trainer.train()